# Resumable Mistral and Qwen3 experiment sweep

Run this notebook top to bottom on a Colab T4. The plan below contains the four Mistral settings already tried and three Qwen3-8B settings while Llama access is pending. A run with the same model, epochs, learning rate, and comparison fingerprint is skipped when it already has a completed row in the Drive CSV. Each new run uses the existing Phase 1 pipeline, cached teacher labels, and a unique Drive output directory. Results are saved after every run so a Colab disconnect does not erase completed experiments.

The final execution cell launches the remaining experiments and makes judge API calls. Review the printed queue before running it. No Git push occurs.

In [ ]:
# 1. Sweep plan. Edit this list once to add or remove experiments.
BRANCH_NAME = 'ak'
DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
REPO_URL = 'https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git'
MISTRAL = 'mistralai/Mistral-7B-Instruct-v0.3'
QWEN = 'Qwen/Qwen3-8B'
SWEEP = [
    {'model_id': MISTRAL, 'epochs': 3, 'learning_rate': 2e-4, 'run_name': 'mistral-7b-e3-lr2e4'},
    {'model_id': MISTRAL, 'epochs': 5, 'learning_rate': 2e-4, 'run_name': 'mistral-7b-e5-lr2e4'},
    {'model_id': MISTRAL, 'epochs': 5, 'learning_rate': 1e-4, 'run_name': 'mistral-7b-e5-lr1e4'},
    {'model_id': MISTRAL, 'epochs': 8, 'learning_rate': 1e-4, 'run_name': 'mistral-7b-e8-lr1e4'},
    {'model_id': QWEN, 'epochs': 3, 'learning_rate': 2e-4, 'run_name': 'qwen3-8b-e3-lr2e4'},
    {'model_id': QWEN, 'epochs': 5, 'learning_rate': 2e-4, 'run_name': 'qwen3-8b-e5-lr2e4'},
    {'model_id': QWEN, 'epochs': 5, 'learning_rate': 1e-4, 'run_name': 'qwen3-8b-e5-lr1e4'},
]
assert BRANCH_NAME not in ('', 'main', 'master')
assert all(x['model_id'] in (MISTRAL, QWEN) and isinstance(x['epochs'], int) and x['epochs'] > 0 and 0 < x['learning_rate'] < 1 for x in SWEEP)
assert len({(x['model_id'], x['epochs'], x['learning_rate']) for x in SWEEP}) == len(SWEEP), 'Duplicate settings in sweep.'
print(f'{len(SWEEP)} configurations in plan; completed ones will be skipped.')

In [ ]:
# 2. GPU and Drive.
import os, sys, gc, subprocess
from pathlib import Path
import torch
from google.colab import drive
assert torch.cuda.is_available(), 'Select a T4 GPU runtime and reconnect.'
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory / 2**30 >= 14, 'At least about 15 GiB GPU memory is needed.'
print(f'GPU: {gpu.name}, {gpu.total_memory/2**30:.1f} GiB')
drive.mount('/content/drive')
drive_root = Path(DRIVE_ROOT)
assert drive_root.parent.exists(), 'Drive did not mount.'
(drive_root / 'experiments').mkdir(parents=True, exist_ok=True)
print('Drive:', drive_root)

In [ ]:
# 3. Clone your branch. PAT stays in environment, never in the Git URL.
import base64
from urllib.parse import urlsplit
from google.colab import userdata
pat = userdata.get('GITHUB_PAT')
assert pat, 'Add GITHUB_PAT to Colab Secrets and enable notebook access.'
git_env = os.environ.copy()
auth = base64.b64encode(('x-access-token:' + pat).encode()).decode()
git_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
               GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' + auth, GIT_TERMINAL_PROMPT='0')
del pat, auth
repo_dir = Path('/content/project')
def git(*args, capture=False):
    return subprocess.run(['git', *args], cwd=repo_dir if repo_dir.exists() else None,
                          env=git_env, text=True, capture_output=capture, check=True)
if repo_dir.exists():
    assert (repo_dir / '.git').is_dir(), '/content/project is not a Git clone.'
    remote = git('remote', 'get-url', 'origin', capture=True).stdout.strip()
    parsed = urlsplit(remote)
    assert parsed.hostname == 'github.com' and parsed.path.rstrip('/') == urlsplit(REPO_URL).path.rstrip('/'), 'Unexpected origin URL.'
    if remote != REPO_URL: git('remote', 'set-url', 'origin', REPO_URL)
    assert git('branch', '--show-current', capture=True).stdout.strip() == BRANCH_NAME, 'Wrong checked-out branch.'
    if git('status', '--porcelain', capture=True).stdout.strip():
        print('Existing clone has local changes; using it without pulling. The memory patch cell below is idempotent.')
    else:
        git('pull', '--ff-only', 'origin', BRANCH_NAME)
else:
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH_NAME, REPO_URL, str(repo_dir)],
                   env=git_env, check=True)
code_dir = repo_dir / 'code'
assert (code_dir / 'main.py').is_file(), 'Missing code/main.py.'
commit_sha = git('rev-parse', 'HEAD', capture=True).stdout.strip()
print(f'Branch {BRANCH_NAME} at {commit_sha[:12]}')

In [ ]:
# 4. Same dependency order as the setup notebook.
for filename in ('requirements.txt', 'requirements_colab.txt'):
    req = code_dir / filename
    assert req.is_file(), f'Missing {req}'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=True)
print('Dependencies installed.')

In [ ]:
# 5. Secrets. Values are not displayed or written to Drive.
for name in ('ANTHROPIC_API_KEY', 'HF_TOKEN'):
    value = userdata.get(name)
    assert value, f'Add {name} to Colab Secrets and enable notebook access.'
    os.environ[name] = value
optional_openai = userdata.get('OPENAI_API_KEY')
if optional_openai: os.environ['OPENAI_API_KEY'] = optional_openai
os.environ['HF_HOME'] = str(drive_root / 'hf_cache')
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
print('API and Hugging Face credentials loaded (values hidden).')

In [ ]:
# 6. Apply T4 memory and Qwen3 chat-template fixes to this Colab checkout.
# This is idempotent and changes no files on GitHub.
pipeline_path = code_dir / 'phase1/pipeline.py'
trainer_path = code_dir / 'phase1/finetuning/trainer.py'
dataset_path = code_dir / 'phase1/finetuning/dataset.py'
templates_path = code_dir / 'phase1/prompts/templates.py'
def replace_once(text, old, new, label, already=None):
    if old in text:
        assert text.count(old) == 1, f'Unexpected multiple matches: {label}'
        return text.replace(old, new, 1)
    if (already or new) in text: return text
    raise RuntimeError(f'Could not safely patch {label}; inspect the branch version.')
pipeline_text = pipeline_path.read_text()
trainer_text = trainer_path.read_text()
dataset_text = dataset_path.read_text()
templates_text = templates_path.read_text()
pipeline_text = replace_once(
    pipeline_text,
    '        # We need a tokenizer for dataset construction; load a temp one\n'
    '        _, tokenizer_tmp = load_model_and_tokenizer(cfg)',
    '        # Dataset construction needs only the chat template and token counter.\n'
    '        from transformers import AutoTokenizer\n'
    '        tokenizer_tmp = AutoTokenizer.from_pretrained(\n'
    '            cfg["student_slm"]["model_id"].strip(),\n'
    '            cache_dir=paths.get("hf_cache"),\n'
    '        )', 'tokenizer-only dataset preparation', already='Dataset construction needs only the chat template')
pipeline_text = replace_once(pipeline_text, '            del base_model\n',
                             '            del base_model, base_tok\n', 'baseline cleanup')
pipeline_text = replace_once(pipeline_text, '                del ft_model\n',
                             '                del ft_model, ft_base, ft_tok\n', 'fine-tuned cleanup')
trainer_text = replace_once(
    trainer_text, '        bnb_4bit_compute_dtype=torch.bfloat16,\n',
    '        bnb_4bit_compute_dtype=(\n'
    '            torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n'
    '        ),\n', 'T4 FP16 fallback')
dataset_text = replace_once(
    dataset_text,
    '            text = tokenizer.apply_chat_template(\n'
    '                full_messages,\n'
    '                tokenize=False,\n'
    '                add_generation_prompt=False,\n'
    '            )',
    '            # Qwen3 defaults to thinking; train on direct labels instead.\n'
    '            template_kwargs = {\"enable_thinking\": False} if cfg[\"student_slm\"][\"model_id\"].lower().startswith(\"qwen/qwen3-\") else {}\n'
    '            text = tokenizer.apply_chat_template(\n'
    '                full_messages,\n'
    '                tokenize=False,\n'
    '                add_generation_prompt=False,\n'
    '                **template_kwargs,\n'
    '            )', 'Qwen3 non-thinking training template', already='Qwen3 defaults to thinking; train on direct labels instead.')
templates_text = replace_once(
    templates_text,
    '    prompt   = tokenizer.apply_chat_template(\n'
    '        messages,\n'
    '        tokenize=False,\n'
    '        add_generation_prompt=True,   # adds the opening of the assistant turn\n'
    '    )',
    '    # Match Qwen3 inference to its non-thinking training examples.\n'
    '    template_kwargs = {\"enable_thinking\": False} if cfg[\"student_slm\"][\"model_id\"].lower().startswith(\"qwen/qwen3-\") else {}\n'
    '    prompt   = tokenizer.apply_chat_template(\n'
    '        messages,\n'
    '        tokenize=False,\n'
    '        add_generation_prompt=True,   # adds the opening of the assistant turn\n'
    '        **template_kwargs,\n'
    '    )', 'Qwen3 non-thinking inference template', already='Match Qwen3 inference to its non-thinking training examples.')
pipeline_path.write_text(pipeline_text)
trainer_path.write_text(trainer_text)
dataset_path.write_text(dataset_text)
templates_path.write_text(templates_text)
assert 'Dataset construction needs only the chat template' in pipeline_text
assert 'tokenizer_tmp = AutoTokenizer.from_pretrained(' in pipeline_text
assert 'del ft_model, ft_base, ft_tok' in pipeline_text
assert 'torch.cuda.is_bf16_supported() else torch.float16' in trainer_text
assert 'Qwen3 defaults to thinking; train on direct labels instead.' in dataset_text
assert 'Match Qwen3 inference to its non-thinking training examples.' in templates_text
print('T4 memory and Qwen3 non-thinking template fixes present in this Colab checkout.')

In [ ]:
# 7. Check cached data, shared comparison settings, and model access.
import json, csv, re, uuid, time, hashlib, yaml, pandas as pd
from datetime import datetime, timezone
from huggingface_hub import model_info, hf_hub_download, whoami
from packaging.version import Version
import transformers
source_cfg_path = code_dir / 'configs/phase1_config.yaml'
source_cfg = yaml.safe_load(source_cfg_path.read_text())
assert source_cfg['teacher_llm']['provider'] == 'anthropic' and source_cfg['teacher_llm']['model'] == 'claude-haiku-4-5', 'Teacher config changed.'
processed = drive_root / 'data/processed'
needed = ('bitext_clustered.csv', 'bitext_grouped.csv', 'bitext_labeled.csv')
missing = [name for name in needed if not (processed / name).is_file()]
assert not missing, f'Missing cached data {missing}. Run the setup notebook once.'
labeled_path = processed / 'bitext_labeled.csv'
labeled = pd.read_csv(labeled_path)
teacher_tag = source_cfg['teacher_llm']['model'].replace('/', '_').replace('-', '_').replace('.', '_').replace(' ', '_')
label_columns = [f'cluster_name_{teacher_tag}_P{i}' for i in range(1, 6)]
assert all(c in labeled.columns for c in label_columns) and labeled[label_columns].notna().all().all(), 'Cached teacher labels are incomplete or mismatched.'
dataset_hash = hashlib.sha256(labeled_path.read_bytes()).hexdigest()
comparison = {key: source_cfg[key] for key in ('seed','dataset','clustering','top_k','teacher_llm','prompts','split','evaluation')}
comparison['labeled_sha256'] = dataset_hash
comparison_hash = hashlib.sha256(json.dumps(comparison, sort_keys=True).encode()).hexdigest()
results_csv = drive_root / 'experiments/experiment_results.csv'
if results_csv.exists():
    old_results = pd.read_csv(results_csv)
    completed = old_results[old_results['status'] == 'completed']
    assert completed.empty or set(completed['comparison_hash'].dropna()) == {comparison_hash}, 'Dataset or evaluation settings differ from earlier completed runs.'
else:
    old_results = pd.DataFrame()
def run_key(item):
    return (item['model_id'], int(item['epochs']), round(float(item['learning_rate']), 10))
done_keys = set()
if not old_results.empty:
    done_keys = {(str(r.model_id), int(r.epochs), round(float(r.learning_rate), 10))
                 for r in old_results.itertuples() if r.status == 'completed' and r.comparison_hash == comparison_hash}
queue = [item for item in SWEEP if run_key(item) not in done_keys]
if any(item['model_id'] == QWEN for item in queue):
    assert Version(transformers.__version__) >= Version('4.51.0'), f'Qwen3 needs transformers>=4.51.0; found {transformers.__version__}'
    assert 'Qwen3 defaults to thinking; train on direct labels instead.' in dataset_path.read_text()
    assert 'Match Qwen3 inference to its non-thinking training examples.' in templates_path.read_text()
print('Completed settings skipped:', len(SWEEP) - len(queue))
print('Queued settings:')
for item in queue: print(' ', item)
if queue: print('HF_TOKEN account:', whoami(token=os.environ['HF_TOKEN'])['name'])
for model_id in sorted({item['model_id'] for item in queue}):
    try:
        info = model_info(model_id, token=os.environ['HF_TOKEN'])
        hf_hub_download(repo_id=model_id, filename='config.json', token=os.environ['HF_TOKEN'], force_download=True)
        if model_id == QWEN:
            tokenizer_check = transformers.AutoTokenizer.from_pretrained(model_id, token=os.environ['HF_TOKEN'])
            tokenizer_check.apply_chat_template([{'role':'user','content':'Give a short theme label.'}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
            del tokenizer_check
        print('Model found:', info.id, '| gated:', info.gated)
    except Exception as exc:
        raise RuntimeError(f'Cannot download {model_id}/config.json with HF_TOKEN. If Llama, request/accept access at https://huggingface.co/{model_id} while signed in as the HF_TOKEN account, then rerun this cell. {type(exc).__name__}: {exc}') from exc

### Review before launch

Only the printed queue will run. Completed Mistral runs are identified by model, epochs, and actual recorded learning rate, so the mislabeled 5-epoch 1e-4 row is still skipped. The sweep uses batch size 1, accumulation 16, 4-bit QLoRA, FP16, gradient checkpointing, and 1024 tokens for both models. It stops if these settings cause examples to be skipped, if a run fails, or if the GPU runs out of memory.

In [ ]:
# 8. Reusable controller around the existing Phase 1 pipeline.
def patch_yaml_line(text, section, key, value):
    lines = text.splitlines(keepends=True)
    starts = [i for i, line in enumerate(lines) if re.match(r'^' + re.escape(section) + r':(?:\s*#.*)?\s*$', line)]
    assert len(starts) == 1, f'Expected one YAML section {section}'
    start = starts[0] + 1
    end = next((i for i in range(start, len(lines)) if re.match(r'^[A-Za-z_][\w-]*:', lines[i])), len(lines))
    hits = [i for i in range(start, end) if re.match(r'^  ' + re.escape(key) + r':', lines[i])]
    assert len(hits) == 1, f'Expected one {section}.{key}'
    i = hits[0]
    match = re.match(r'^(  ' + re.escape(key) + r':\s*)([^#\n]*?)(\s*(?:#.*)?)(\n?)$', lines[i])
    assert match, f'Cannot safely edit {section}.{key}'
    lines[i] = match.group(1) + json.dumps(value) + match.group(3) + match.group(4)
    return ''.join(lines)

def already_completed(item):
    if not results_csv.exists(): return False
    table = pd.read_csv(results_csv)
    rows = table[(table['status'] == 'completed') & (table['comparison_hash'] == comparison_hash)]
    return any((str(r.model_id), int(r.epochs), round(float(r.learning_rate), 10)) == run_key(item)
               for r in rows.itertuples())

def make_run_config(item):
    name = item['run_name']
    assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', name), 'Invalid run_name slug.'
    experiment_id = f'{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{name}_{uuid.uuid4().hex[:8]}'
    experiment_dir = drive_root / 'experiments' / experiment_id
    experiment_dir.mkdir(parents=True, exist_ok=False)
    runs_root = experiment_dir / 'runs'
    runs_root.mkdir()
    overrides = {
        ('student_slm','model_id'): item['model_id'], ('student_slm','max_seq_length'): 1024,
        ('training','num_train_epochs'): item['epochs'], ('training','learning_rate'): float(item['learning_rate']),
        ('training','per_device_train_batch_size'): 1, ('training','gradient_accumulation_steps'): 16,
        ('training','gradient_checkpointing'): True, ('training','bf16'): False, ('training','fp16'): True,
        ('qlora','load_in_4bit'): True, ('qlora','use_double_quant'): True,
        ('paths','outputs'): str(runs_root),
        ('pipeline','run_clustering'): False, ('pipeline','run_preprocessing'): False,
        ('pipeline','run_label_generation'): False, ('pipeline','run_finetuning'): True,
        ('pipeline','run_baseline_eval'): True, ('pipeline','run_finetuned_eval'): True,
        ('pipeline','run_llm_judge'): True, ('pipeline','run_business_eval'): True,
        ('evaluation','existing_run_dir'): None, ('evaluation','eval_all_splits'): False,
    }
    config_text = source_cfg_path.read_text()
    for (section, key), value in overrides.items():
        config_text = patch_yaml_line(config_text, section, key, value)
    run_cfg = yaml.safe_load(config_text)
    for (section, key), value in overrides.items(): assert run_cfg[section][key] == value
    run_cfg_path = experiment_dir / 'phase1_config.yaml'
    run_cfg_path.write_text(config_text)
    manifest = dict(experiment_id=experiment_id, **item, branch=BRANCH_NAME, commit_sha=commit_sha,
                    comparison_hash=comparison_hash, labeled_sha256=dataset_hash,
                    overrides={f'{s}.{k}':v for (s,k),v in overrides.items()})
    (experiment_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return experiment_id, experiment_dir, runs_root, run_cfg_path, run_cfg

def collect_result(item, experiment_id, experiment_dir, runs_root, log_path, exit_code, oom, wall_time_s):
    run_dirs = [p for p in runs_root.iterdir() if p.is_dir()]
    assert len(run_dirs) <= 1, f'Unexpected multiple run directories: {run_dirs}'
    pipeline_run_dir = run_dirs[0] if run_dirs else None
    eval_dir = pipeline_run_dir / 'evaluation' if pipeline_run_dir else None
    def test_value(filename, column):
        path = eval_dir / filename if eval_dir else None
        if not path or not path.exists(): return None
        table = pd.read_csv(path)
        if not {'split','model',column}.issubset(table.columns): return None
        selected = table[(table['split'] == 'test') & (table['model'] == 'finetuned')]
        if selected.empty: return None
        value = pd.to_numeric(selected.iloc[0][column], errors='coerce')
        return float(value) if pd.notna(value) else None
    def business_value(metric):
        path = eval_dir / 'business_eval.csv' if eval_dir else None
        if not path or not path.exists(): return None
        table = pd.read_csv(path)
        values = table.loc[table['metric'] == metric, 'value']
        return float(values.iloc[0]) if not values.empty else None
    def validation_loss():
        if not pipeline_run_dir: return None
        states = list((pipeline_run_dir / 'models/lora_adapter').glob('checkpoint-*/trainer_state.json'))
        values = [float(entry['eval_loss']) for state in states
                  for entry in json.loads(state.read_text()).get('log_history', []) if 'eval_loss' in entry]
        return min(values) if values else None
    score = test_value('judge_summary.csv', 'composite')
    cosine = test_value('metrics_summary.csv', 'cosine_sim_same')
    rouge = test_value('metrics_summary.csv', 'rouge_l_same')
    split_counts = {part: sum(1 for line in (processed / f'{part}.jsonl').open() if line.strip())
                    if (processed / f'{part}.jsonl').exists() else 0 for part in ('train','val','test')}
    expected_examples = labeled['cluster_id'].nunique() * 5
    all_examples_kept = sum(split_counts.values()) == expected_examples
    status = 'oom' if oom else ('completed' if exit_code == 0 and score is not None and cosine is not None and rouge is not None and all_examples_kept
                             else ('incomparable' if exit_code == 0 and not all_examples_kept else 'failed'))
    row = dict(experiment_id=experiment_id, run_name=item['run_name'], timestamp_utc=datetime.now(timezone.utc).isoformat(),
               status=status, oom=oom, model_id=item['model_id'], epochs=item['epochs'], learning_rate=item['learning_rate'],
               judge_composite_test=score, above_4=(status == 'completed' and score > 4.0),
               cosine_same_test=cosine, rouge_l_same_test=rouge, best_validation_loss=validation_loss(),
               training_time_min=business_value('finetuning_wall_time_min'), wall_time_min=round(wall_time_s/60, 2),
               batch_size=1, gradient_accumulation=16, max_seq_length=1024, qlora_4bit=True,
               train_examples=split_counts['train'], val_examples=split_counts['val'], test_examples=split_counts['test'], expected_examples=expected_examples,
               branch=BRANCH_NAME, commit_sha=commit_sha, comparison_hash=comparison_hash,
               labeled_sha256=dataset_hash, output_dir=str(pipeline_run_dir or ''), log_path=str(log_path))
    (experiment_dir / 'result.json').write_text(json.dumps(row, indent=2))
    return row

def append_result(row):
    fields = list(row)
    if results_csv.exists():
        with results_csv.open(newline='') as f: header = next(csv.reader(f))
        assert header == fields, 'Existing results CSV schema differs; stopped before appending.'
        assert row['experiment_id'] not in set(pd.read_csv(results_csv)['experiment_id']), 'Run already recorded.'
    with results_csv.open('a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if f.tell() == 0: writer.writeheader()
        writer.writerow(row)

def run_one(item):
    if already_completed(item):
        print('SKIP completed:', item['run_name'], run_key(item))
        return None
    gc.collect()
    torch.cuda.empty_cache()
    free_gb, total_gb = torch.cuda.mem_get_info()
    assert free_gb/2**30 >= 12, f'Only {free_gb/2**30:.1f} GiB free; restart the runtime before another run.'
    experiment_id, experiment_dir, runs_root, run_cfg_path, run_cfg = make_run_config(item)
    print('\nSTART', experiment_id)
    print(json.dumps({'model':item['model_id'], 'epochs':item['epochs'], 'learning_rate':item['learning_rate'],
                      'batch_size':1, 'gradient_accumulation':16, 'max_seq_length':1024,
                      'qlora':run_cfg['qlora'], 'split':run_cfg['split'], 'evaluation':run_cfg['evaluation'],
                      'outputs':str(runs_root), 'config':str(run_cfg_path)}, indent=2))
    log_path = experiment_dir / 'pipeline.log'
    cmd = [sys.executable, '-u', 'main.py', '--phase', '1', '--config', str(run_cfg_path), '--device_mode', 'colab']
    t0 = time.monotonic()
    with log_path.open('w') as log, subprocess.Popen(cmd, cwd=code_dir, env=os.environ.copy(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        exit_code = proc.wait()
    elapsed_s = time.monotonic() - t0
    gc.collect()
    torch.cuda.empty_cache()
    log_text = log_path.read_text(errors='replace')
    oom = bool(re.search(r'(out of memory|CUDA error: out of memory|CUBLAS_STATUS_ALLOC_FAILED)', log_text, re.I))
    row = collect_result(item, experiment_id, experiment_dir, runs_root, log_path, exit_code, oom, elapsed_s)
    append_result(row)
    print('\nRECORDED:', row['run_name'], '|', row['status'], '| judge:', row['judge_composite_test'], '| minutes:', row['wall_time_min'])
    if row['status'] != 'completed':
        raise RuntimeError(f'Run {row["status"]}; inspect {log_path}. The sweep stopped without launching another run.')
    return row

In [ ]:
# 9. Launch the remaining runs in order. This cell makes judge API calls.
# Results are appended to Drive after EACH run. Re-running skips completed settings.
if not queue: print('All planned settings are already completed.')
for index, item in enumerate(queue, 1):
    print(f'\nSWEEP {index}/{len(queue)}: {item["run_name"]}')
    run_one(item)
print('Sweep complete. Results:', results_csv)

In [ ]:
# 10. Compare the held-out TEST scores. Judge composite > 4.0 is the target.
from IPython.display import display
results = pd.read_csv(results_csv)
results = results[results['comparison_hash'] == comparison_hash]
columns = ['run_name','model_id','epochs','learning_rate','status','oom',
           'judge_composite_test','above_4','cosine_same_test',
           'rouge_l_same_test','best_validation_loss','training_time_min']
display(results[columns].sort_values('judge_composite_test', ascending=False, na_position='last')
        .style.format({'judge_composite_test':'{:.2f}', 'cosine_same_test':'{:.3f}',
                       'rouge_l_same_test':'{:.3f}', 'best_validation_loss':'{:.3f}',
                       'training_time_min':'{:.1f}'}, na_rep='—')
        .highlight_max(subset=['judge_composite_test'], color='#b8e6c4'))
print('Results CSV:', results_csv)

The teacher's reference-mode 5/5 score compares each label to itself. For model comparisons, use the held-out finetuned judge score, cosine similarity, ROUGE-L, validation loss, and example counts together. A score above 4.0 is a goal, not a guaranteed result. If Colab disconnects, reopen the notebook and run from the top; completed configurations will be skipped.

In [ ]:
# Optional Git review. There is deliberately no push command in this notebook.
subprocess.run(['git', 'status', '--short', '--branch'], cwd=repo_dir, check=True)
subprocess.run(['git', 'diff', '--stat'], cwd=repo_dir, check=True)